# 77 — Q-planning fork pilot: manifest and restoration preflight

This notebook freezes 64 roots for each of three acquisition strategies (random, U20, and
failure), then replays one root per strategy twice. Collection workers should not be launched
unless all three restoration checks pass.

The future training contract is **65% fork-priority / 35% ordinary replay**. This notebook
collects no training branches and does not touch the held-out position-perturbation suites.


## 1. Environment


In [ ]:
EXTRAS = 'sim,analysis'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen(
    'https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/'
    'pnp-vla/scripts/colab_bootstrap.py').read().decode())


## 2. Build and publish the immutable root manifest


In [ ]:
from pathlib import Path
from google.colab import drive
from pnp.qplanning_fork_pilot import (
    FORK_PILOT_MANIFEST_PATH,
    create_fork_pilot_manifest,
)
from pnp.store import SupabaseStore

drive.mount('/content/drive')
store = SupabaseStore()
CACHE_ROOT = Path('/content/drive/MyDrive/pnp_qplanning_forks')

document = create_fork_pilot_manifest(
    store=store,
    cache_root=CACHE_ROOT,
    download_workers=8,
)
print({
    'manifest_path': FORK_PILOT_MANIFEST_PATH,
    'manifest_hash': document['manifest_hash'],
    'trees': len(document['payload']['trees']),
    'strategies': document['payload']['strategies'],
})


## 3. Exact replay/restoration sentinel


In [ ]:
from pnp.qplanning_fork_pilot import run_fork_restoration_preflight

restoration_report = run_fork_restoration_preflight(
    manifest_path=FORK_PILOT_MANIFEST_PATH,
    store=store,
)
assert all(row['passed'] for row in restoration_report)
restoration_report


If the final assertion passes, the parent replay, canonical MuJoCo correction, stock candidate,
and eventual stock-branch outcome were reproducible on one root from every strategy. Launch all
four notebook-78 workers next.
